## [DeiT: Data-efficient Image Transformers](https://arxiv.org/abs/2012.12877)

ViT needs **hundreds of millions of images** (JFT-300M) to train well. Without that data, it significantly underperforms CNNs. DeiT (Touvron et al., 2020) asks: *can we train a competitive ViT using only ImageNet — no extra data?*

The answer is yes, with two key contributions:

1. **Careful training recipe** — strong augmentation (RandAugment, Mixup, CutMix, repeated augmentation), cosine schedule, label smoothing. Turns out ViT just needed better training, not more data.
2. **Knowledge distillation via a distillation token** — a learnable token (like the CLS token) that is trained to mimic the output of a CNN teacher (e.g. RegNet). The student learns not just from labels but from the teacher's soft predictions.

<img src="./figures/deit_compare.png" title="DeiT vs ViT vs EfficientNet" width="420"/>

DeiT-B matches EfficientNet on ImageNet **trained on ImageNet only** — ViT-B trained the same way scores 77.9%. The distilled DeiT-B↑384 reaches 85.2%.

### The Distillation Token

<img src="./figures/deit_distill.png" title="DeiT distillation token" width="320"/>

DeiT adds a **distillation token** alongside the class token. The sequence fed to the Transformer is:

$$[\texttt{cls}] \ | \ p_1, p_2, \ldots, p_N \ | \ [\texttt{dist}]$$

Both tokens interact with all patch tokens through self-attention. At the output:
- **CLS token** → trained with standard cross-entropy loss against the true label  
- **Dist token** → trained to match the **teacher's prediction** (hard or soft label)

**Hard distillation** (default): the distillation target is `argmax(teacher(x))` — just a one-hot label from the teacher.  
**Soft distillation**: use KL-divergence against the teacher's full softmax distribution.

**Why not just use the teacher's labels as ground truth?**  
> The teacher sometimes disagrees with the true label — e.g., it sees a dog as "terrier" while the label says "dog". A separate distillation token lets the model learn both signals simultaneously without forcing them to agree.

**Final prediction** at inference: average the CLS and dist token logits.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

os.environ['http_proxy']  = 'http://192.41.170.23:3128'
os.environ['https_proxy'] = 'http://192.41.170.23:3128'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, random_split

from torchvision.datasets.mnist import MNIST
from torchvision.transforms import ToTensor, Compose, Resize
from torchvision.models import resnet18

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

np.random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device', device)

## Dataset and Dataloader

We use MNIST resized to 32×32. For the distillation demo, a small pretrained-style CNN plays the role of the "teacher".

In [ ]:
transform = Compose([Resize((32, 32)), ToTensor()])

full_dataset = MNIST(root='./data/', train=True,  download=True, transform=transform)
train_size   = int(0.8 * len(full_dataset))
val_size     = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])
test_set = MNIST(root='./data/', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, shuffle=True,  batch_size=64)
val_loader   = DataLoader(val_set,   shuffle=False, batch_size=64)
test_loader  = DataLoader(test_set,  shuffle=False, batch_size=64)

for images, labels in train_loader:
    break

print(images.shape)   # (64, 1, 32, 32)

the_image = images[0].permute(1, 2, 0)
plt.figure(figsize=(2, 2))
plt.imshow(the_image, cmap='gray')
plt.title(f'Label: {labels[0].item()}')
plt.axis('off')
plt.show()

## DeiT Model

DeiT is architecturally identical to ViT — same Transformer encoder — with one change: **two special tokens** instead of one.

We build it step by step.

#### Step 1: Patch Embedding

Same as ViT — split the 32×32 image into 4×4 patches and linearly project each to hidden dim D.

$$(N, 1, 32, 32) \rightarrow (N, 64, D)$$

In [ ]:
class PatchEmbedding(nn.Module):
    """Split image into patches and project to hidden dim."""

    def __init__(self, img_size=32, patch_size=4, in_chans=1, embed_dim=64):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2   # 64
        # Use a Conv2d with kernel=patch_size, stride=patch_size — equivalent to linear projection per patch
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (N, C, H, W) → (N, embed_dim, H/P, W/P) → (N, n_patches, embed_dim)
        x = self.proj(x)                    # (N, D, 8, 8)
        x = x.flatten(2).transpose(1, 2)   # (N, 64, D)
        return x

patch_embed = PatchEmbedding()
tokens = patch_embed(images)
print('Patch tokens:', tokens.shape)   # (64, 64, 64)

#### Step 2: CLS Token + Distillation Token + Positional Embedding

DeiT prepends **two** learnable tokens to the patch sequence:

- `[cls]` — the standard classification token from ViT  
- `[dist]` — the new distillation token, same shape, learns to mimic the teacher

Sequence length: `n_patches + 2` = 66

Both tokens interact with all patch tokens through self-attention — the teacher's signal propagates into the entire representation.

In [ ]:
embed_dim  = 64
n_patches  = 64
seq_len    = n_patches + 2   # cls + patches + dist

cls_token  = nn.Parameter(torch.zeros(1, 1, embed_dim))
dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
pos_embed  = nn.Parameter(torch.zeros(1, seq_len, embed_dim))  # learned positional embedding

nn.init.trunc_normal_(cls_token,  std=0.02)
nn.init.trunc_normal_(dist_token, std=0.02)
nn.init.trunc_normal_(pos_embed,  std=0.02)

# Concatenate: [cls | patches | dist]
N = tokens.shape[0]
cls_expanded  = cls_token.expand(N, -1, -1)    # (N, 1, D)
dist_expanded = dist_token.expand(N, -1, -1)   # (N, 1, D)
x = torch.cat([cls_expanded, tokens, dist_expanded], dim=1)  # (N, 66, D)
x = x + pos_embed
print('Full sequence:', x.shape)   # (64, 66, 64)

#### Step 3: Transformer Encoder

Standard ViT encoder — Multi-head Self-Attention + MLP with residual connections and LayerNorm. DeiT uses the exact same architecture as ViT; the distillation token just rides along as part of the sequence.

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        y = self.norm1(x)
        x = x + self.attn(y, y, y, need_weights=False)[0]
        x = x + self.mlp(self.norm2(x))
        return x

#### Step 4: Full DeiT with Distillation Head

Two output heads on top of the Transformer:
- **`head_cls`**: Linear(D, num_classes) — trained with cross-entropy against ground truth
- **`head_dist`**: Linear(D, num_classes) — trained to match the teacher's prediction

At **inference**, average both logits for the final prediction.

In [ ]:
class DeiT(nn.Module):

    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_chans=1,
        num_classes=10,
        embed_dim=64,
        depth=6,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.1,
    ):
        super().__init__()

        # Patch embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_chans, embed_dim)
        n_patches = self.patch_embed.n_patches

        # Two learnable special tokens
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed  = nn.Parameter(torch.zeros(1, n_patches + 2, embed_dim))
        nn.init.trunc_normal_(self.cls_token,  std=0.02)
        nn.init.trunc_normal_(self.dist_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed,  std=0.02)

        self.pos_drop = nn.Dropout(dropout)

        # Transformer encoder
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Two classification heads
        self.head_cls  = nn.Linear(embed_dim, num_classes)   # trained with CE loss
        self.head_dist = nn.Linear(embed_dim, num_classes)   # trained with distillation loss

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        N = x.shape[0]

        # Patch embed + prepend cls and dist tokens
        x = self.patch_embed(x)   # (N, n_patches, D)
        cls  = self.cls_token.expand(N, -1, -1)
        dist = self.dist_token.expand(N, -1, -1)
        x = torch.cat([cls, x, dist], dim=1)   # (N, n_patches+2, D)
        x = self.pos_drop(x + self.pos_embed)

        # Transformer
        x = self.blocks(x)
        x = self.norm(x)

        # Extract the two special tokens
        cls_out  = x[:, 0]    # CLS token output
        dist_out = x[:, -1]   # dist token output

        if self.training:
            # Return both logits separately during training
            return self.head_cls(cls_out), self.head_dist(dist_out)
        else:
            # At inference: average both heads
            return (self.head_cls(cls_out) + self.head_dist(dist_out)) / 2


model = DeiT(
    img_size=32, patch_size=4, in_chans=1, num_classes=10,
    embed_dim=64, depth=6, num_heads=4,
)
model = model.to(device)
print(model)

In [ ]:
# Sanity check
model.eval()
with torch.no_grad():
    out = model(torch.zeros(2, 1, 32, 32).to(device))
    print('Inference output:', out.shape)   # (2, 10) — averaged from both heads

model.train()
with torch.no_grad():
    cls_out, dist_out = model(torch.zeros(2, 1, 32, 32).to(device))
    print('Train CLS  output:', cls_out.shape)   # (2, 10)
    print('Train Dist output:', dist_out.shape)  # (2, 10)

## Teacher Model

In the paper, the teacher is a RegNet or EfficientNet CNN. Here we use a lightweight CNN teacher trained on MNIST. The teacher is **frozen** — it only provides soft targets.

We first train the teacher, then use it to distill into DeiT.

In [ ]:
class CNNTeacher(nn.Module):
    """Simple CNN teacher for MNIST."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),   # 16x16
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),   # 8x8
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


teacher = CNNTeacher().to(device)

# Train teacher
teacher_optimizer = torch.optim.Adam(teacher.parameters(), lr=1e-3)
criterion_ce = CrossEntropyLoss()

print('Training teacher CNN...')
teacher.train()
for epoch in range(3):
    correct, total, loss_sum = 0, 0, 0.0
    for x, y in tqdm(train_loader, desc=f'Teacher Epoch {epoch+1}'):
        x, y = x.to(device), y.to(device)
        logits = teacher(x)
        loss   = criterion_ce(logits, y)
        teacher_optimizer.zero_grad()
        loss.backward()
        teacher_optimizer.step()
        loss_sum += loss.item()
        correct  += (logits.argmax(1) == y).sum().item()
        total    += len(y)
    print(f'  Epoch {epoch+1}: loss={loss_sum/len(train_loader):.4f}, acc={correct/total*100:.2f}%')

# Freeze teacher
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print('Teacher frozen.')

## Training DeiT with Knowledge Distillation

The total loss combines two objectives:

$$\mathcal{L} = (1 - \lambda)\,\mathcal{L}_{\text{CE}}(y_{\text{cls}}, y) + \lambda\,\mathcal{L}_{\text{CE}}(y_{\text{dist}},\ \hat{y}_{\text{teacher}})$$

where $\hat{y}_{\text{teacher}} = \arg\max(\text{teacher}(x))$ is the hard teacher label (default in DeiT).  
$\lambda = 0.5$ by default — equal weight to both losses.

In [ ]:
import torch.optim as optim

num_epochs  = 5
lr          = 1e-3
lambda_dist = 0.5   # weight on the distillation loss

optimizer = optim.Adam(model.parameters(), lr=lr)
criterion_ce = CrossEntropyLoss()

In [ ]:
def train_deit(model, teacher, loader, optimizer, device, lambda_dist=0.5):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device)

        # Teacher hard labels (no grad needed — teacher is frozen)
        with torch.no_grad():
            teacher_labels = teacher(x).argmax(dim=1)

        # DeiT forward returns (cls_logits, dist_logits) during training
        cls_logits, dist_logits = model(x)

        loss_cls  = criterion_ce(cls_logits,  y)
        loss_dist = criterion_ce(dist_logits, teacher_labels)
        loss      = (1 - lambda_dist) * loss_cls + lambda_dist * loss_dist

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        # Accuracy from the average of both heads
        avg_logits = (cls_logits + dist_logits) / 2
        correct    += (avg_logits.argmax(1) == y).sum().item()
        total      += len(y)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in tqdm(loader):
            x, y = x.to(device), y.to(device)
            logits = model(x)   # averaged at inference
            total_loss += criterion_ce(logits, y).item()
            correct    += (logits.argmax(1) == y).sum().item()
            total      += len(y)
    return total_loss / len(loader), correct / total


def epoch_time(start, end):
    elapsed = end - start
    return int(elapsed // 60), int(elapsed % 60)

In [ ]:
import os
os.makedirs('models', exist_ok=True)

best_valid_loss = float('inf')
save_path       = 'models/DeiT.pt'

train_losses, valid_losses = [], []

for epoch in range(num_epochs):
    start_time = time.time()

    train_loss, train_acc = train_deit(model, teacher, train_loader, optimizer, device, lambda_dist)
    valid_loss, valid_acc = evaluate(model, val_loader, device)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    epoch_mins, epoch_secs = epoch_time(start_time, time.time())

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), save_path)

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%')
    print(f'  Val.  Loss: {valid_loss:.4f} | Val.  Acc: {valid_acc*100:.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(train_losses, label='train loss')
ax.plot(valid_losses, label='valid loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load(save_path, map_location=device))
test_loss, test_acc = evaluate(model, test_loader, device)
print(f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc*100:.2f}%')

## CLS vs Dist Token Comparison

We can inspect how much the two heads agree — and how the distillation head differs from the classification head.

In [ ]:
model.eval()
all_cls, all_dist, all_labels = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        # Temporarily switch to training mode to get both heads
        model.train()
        cls_logits, dist_logits = model(x)
        model.eval()

        all_cls.append(cls_logits.argmax(1).cpu())
        all_dist.append(dist_logits.argmax(1).cpu())
        all_labels.append(y)

all_cls    = torch.cat(all_cls)
all_dist   = torch.cat(all_dist)
all_labels = torch.cat(all_labels)

acc_cls  = (all_cls  == all_labels).float().mean().item()
acc_dist = (all_dist == all_labels).float().mean().item()
agree    = (all_cls  == all_dist).float().mean().item()

print(f'CLS  head accuracy : {acc_cls*100:.2f}%')
print(f'Dist head accuracy : {acc_dist*100:.2f}%')
print(f'Heads agree on     : {agree*100:.2f}% of test samples')